# ThreadVault DSA Demonstration

This notebook imports the implementations in `dsa/`; it does not define substitute versions. Run it with the project root as the working directory. The marketplace's Python API is the application layer, MariaDB persists marketplace state, and these modules provide the specific data structures and algorithms used by selected workflows.

Before running each example, **predict the output** and explain which operation determines it. Then run the cell and compare your reasoning with the result.

In [ ]:
from pathlib import Path
from timeit import repeat

from dsa.priority_queue import PriorityQueue
from dsa.queue import Queue
from dsa.searching import binary_search, linear_search, linear_search_all
from dsa.sorting import insertion_sort, merge_sort
from dsa.stack import Stack

if not (Path.cwd() / "dsa").is_dir():
    raise RuntimeError("Run this notebook with the ThreadVault project root as the working directory.")

"Imports point to the same source files used by the application and tests."

## 1. Queue: FIFO checkout processing

The linked `Queue` removes items in the same order they were added: first in, first out (FIFO). In ThreadVault, the API uses it to validate checkout line items in the order submitted **during that request**. It is not a persistent order queue in MariaDB.

In [ ]:
checkout_lines = Queue()
for line in ["shirt / M", "pants / 32", "cap / one size"]:
    checkout_lines.enqueue(line)

processed_in_order = []
while not checkout_lines.is_empty():
    processed_in_order.append(checkout_lines.dequeue())

processed_in_order

In [ ]:
empty_queue = Queue()
try:
    empty_queue.dequeue()
except IndexError as error:
    print(type(error).__name__, str(error))

## 2. Stack: undoing the latest status change

A stack is last in, first out (LIFO). The app stores status history in MariaDB and rebuilds a Stack when it handles undo; the in-memory Stack itself is not durable storage. Undo applies to the most recent seller status change, not to stock deductions.

In [ ]:
status_changes = Stack()
status_changes.push(("item-17", "pending", "processing"))
status_changes.push(("item-22", "pending", "processing"))

latest_change_to_undo = status_changes.pop()
latest_change_to_undo, status_changes.peek(), status_changes.size()

## 3. Priority queue: ordering seller work

This is a stable binary **min-heap**. Smaller numeric priorities are returned first. An insertion sequence number breaks ties, so items with equal priorities retain insertion order. ThreadVault maps `urgent`, `express`, and `standard` to ascending numbers. The heap orders work for display; the seller still performs each status update.

In [ ]:
priority_rank = {"urgent": 1, "express": 2, "standard": 3}
seller_work = PriorityQueue()
for item, priority in [("ordinary order", "standard"), ("urgent A", "urgent"), ("urgent B", "urgent"), ("express order", "express")]:
    seller_work.insert(item, priority_rank[priority])

ordered_work = []
while not seller_work.is_empty():
    ordered_work.append(seller_work.extract_min())

ordered_work

## 4. Sorting: catalog order and comparison

Both functions return a **new list** and leave the input list unchanged. Both are stable for equal keys. The marketplace API uses `merge_sort` for product-name ordering; `insertion_sort` is implemented and tested as a second algorithm, but is not used on the main request path.

In [ ]:
products = [("T-02", "Thread tee"), ("T-01", "Thread tee"), ("J-03", "Jacket")]
by_name_merge = merge_sort(products, key=lambda product: product[1])
by_name_insertion = insertion_sort(products, key=lambda product: product[1])

{
    "original_unchanged": products,
    "merge_sort": by_name_merge,
    "insertion_sort": by_name_insertion,
    "equal_name_order_preserved": [product for product in by_name_merge if product[1] == "Thread tee"] == [product for product in products if product[1] == "Thread tee"],
}

## 5. Searching: linear versus binary

`linear_search` checks values in sequence using a predicate and returns the first matching index, or `-1`. `linear_search_all` returns all matching values. `binary_search` returns an index or `-1`, but its input must already be sorted in ascending order by the selected key; otherwise the result is not reliable.

In [ ]:
sku_values = ["A-100", "A-120", "B-200", "C-300", "C-301"]
matching_index = linear_search(sku_values, lambda sku: sku.startswith("B-"))
all_c_skus = linear_search_all(sku_values, lambda sku: sku.startswith("C-"))
binary_index = binary_search(sku_values, "C-300")
missing_index = binary_search(sku_values, "D-400")

matching_index, all_c_skus, binary_index, missing_index

## 6. Complexity reference

Bounds below use `n` for the number of stored values and assume constant-time comparisons/keys. They describe the algorithms, not total HTTP or database request time.

| Operation | Time | Additional space | Notes |
|---|---:|---:|---|
| Linked Queue enqueue / dequeue / peek | O(1) each | O(n) total structure | Head and tail references avoid traversing the list. |
| Linked Stack push / pop / peek | O(1) each | O(n) total structure | Operations use the top node. |
| Binary min-heap insert / extract-min | O(log n) | O(n) total heap | Sifting may traverse heap height. |
| Binary min-heap peek | O(1) | O(n) total heap | Reads the root. |
| Merge sort | O(n log n) | O(n) auxiliary, plus recursion | This implementation copies slices and returns a new list. |
| Insertion sort | O(n²) worst; O(n) best | O(n) for returned copy | The sorting work is in-place on that copy. |
| Linear search / search all | O(n) | O(1) / O(k) output | `k` is number of matches for search-all. |
| Binary search | O(log n) | O(1) | Requires sorted input. |

These are asymptotic bounds. They describe growth as input size grows; they are not promises about a particular machine's elapsed time.

In [ ]:
def best_seconds_per_call(function, number=1, repeats=3):
    return min(repeat(function, number=number, repeat=repeats)) / number


timing_rows = []
for size in [250, 500, 1000, 2000]:
    descending = list(range(size, 0, -1))
    ascending = list(range(size))
    timing_rows.append({
        "n": size,
        "merge_sort_seconds": best_seconds_per_call(lambda values=descending: merge_sort(values)),
        "insertion_sort_reverse_seconds": best_seconds_per_call(lambda values=descending: insertion_sort(values)),
        "linear_search_last_seconds": best_seconds_per_call(lambda values=ascending: linear_search(values, lambda value: value == len(values) - 1)),
        "binary_search_last_seconds": best_seconds_per_call(lambda values=ascending: binary_search(values, len(values) - 1)),
    })

timing_rows

**Interpretation:** These measurements exercise the imported project functions on the current machine. They are empirical and noisy: interpreter warm-up, operating-system scheduling, hardware, and input shape affect elapsed time. A timing sample does not prove a complexity bound; the bounds come from analyzing the algorithm's operations. Reverse-sorted input is a worst-case-shaped example for insertion sort.

## 7. Self-check

Before opening the answers, predict: (1) the order returned by the priority queue when two items tie, (2) which status change a Stack pops first, and (3) why binary search on an unsorted list is not guaranteed to find a present value.

<details><summary>Self-check answers</summary>

1. The tied items come out in the same order they were inserted because the heap stores a sequence number with each priority.
2. The most recently pushed status change is popped first (LIFO).
3. Binary search discards half the remaining range based on sorted-order comparisons. If the values are not sorted by the same key, that elimination step is unsound.

</details>